# Solution: Linear Regression for Economics

**Main application:** Advertising expenditure → Sales revenue (managerial / industrial economics).  
**Secondary:** Simple Keynesian consumption function (estimate MPC).

**Methods:** From-scratch gradient descent (centered), scikit-learn, closed-form OLS, residual diagnostics, Monte-Carlo sensitivity.

---

## Project Flowchart

![Flowchart](economics_linear_regression_flowchart.png)

### Audience adaptation (Economics)
- Technical / econometric readers → gradients, residual structure, estimator comparison.
- Marketing or finance executives → marginal return to advertising + R² in one sentence.
- Macro / policy audience (consumption practice) → MPC interpretation and caveats.
- Non-specialists → rising scatter line only.


## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Load & Explore Advertising → Sales

In [ ]:
df = pd.read_csv('data/economics_ad_sales.csv')
print('Shape:', df.shape)
display(df.head())
display(df.describe().round(2))

plt.figure(figsize=(7,5))
plt.scatter(df['advertising'], df['sales'], alpha=0.55, edgecolor='k', linewidth=0.3, c='#2980B9')
plt.xlabel('Advertising spend ($000)')
plt.ylabel('Sales revenue ($000)')
plt.title('Managerial Economics: Advertising → Sales (n=200)')
plt.tight_layout(); plt.show()

## 2. Points and Lines — Manual Prediction (warm-up)

In [ ]:
quarters = list(range(1, 9))
ad_spend = [20, 28, 35, 42, 50, 55, 62, 70]
sales_obs = [95, 120, 140, 165, 185, 200, 230, 250]

m, b = 3.0, 35
y_pred = [m * x + b for x in ad_spend]

plt.figure(figsize=(7,4))
plt.plot(ad_spend, sales_obs, 'o', label='Observed sales')
plt.plot(ad_spend, y_pred, '-', label=f'Manual line (m={m}, b={b})')
plt.xlabel('Advertising ($000)'); plt.ylabel('Sales ($000)')
plt.title('Quarterly Ad → Sales — Manual Line')
plt.legend(); plt.tight_layout(); plt.show()

print(f'Implied marginal return to advertising: ${m}k sales per $1k ad spend')

## 3. Loss — Sum of Squared Errors

In [ ]:
x = [10, 20, 30]
y = [80, 110, 160]
m1, b1 = 3, 40
m2, b2 = 4, 20

y_pred1 = [m1*xi + b1 for xi in x]
y_pred2 = [m2*xi + b2 for xi in x]

total_loss1 = sum((yi - yh)**2 for yi, yh in zip(y, y_pred1))
total_loss2 = sum((yi - yh)**2 for yi, yh in zip(y, y_pred2))
print('Loss1:', total_loss1, 'Loss2:', total_loss2)
better_fit = 1 if total_loss1 < total_loss2 else 2
print('Better fit: line', better_fit)

# Vectorised alternate
print('Vectorised:', np.sum((np.array(y)-np.array(y_pred1))**2),
      np.sum((np.array(y)-np.array(y_pred2))**2))

## 4–5. Gradients for Intercept and Slope

In [ ]:
def get_gradient_at_b(x, y, m, b):
    N = len(x)
    diff = sum(y[i] - (m*x[i] + b) for i in range(N))
    return -2.0 / N * diff

def get_gradient_at_m(x, y, m, b):
    N = len(x)
    diff = sum(x[i] * (y[i] - (m*x[i] + b)) for i in range(N))
    return -2.0 / N * diff

# Vectorised alternates (preferred in practice)
def get_gradient_at_b_vec(x, y, m, b):
    return -2.0 * np.mean(y - (m*x + b))

def get_gradient_at_m_vec(x, y, m, b):
    return -2.0 * np.mean(x * (y - (m*x + b)))

print('Loop  gb:', get_gradient_at_b([10,20,30],[80,110,160],3,40))
print('Vec   gb:', get_gradient_at_b_vec(np.array([10.,20,30]), np.array([80.,110,160]), 3, 40))

## 6. One Gradient Step

In [ ]:
def step_gradient(x, y, b_current, m_current, learning_rate):
    b_grad = get_gradient_at_b(x, y, m_current, b_current)
    m_grad = get_gradient_at_m(x, y, m_current, b_current)
    b = b_current - learning_rate * b_grad
    m = m_current - learning_rate * m_grad
    return [b, m]

print('One step from (0,0):', step_gradient([10,20,30], [80,110,160], 0, 0, 0.001))

## 7. Full Gradient Descent Loop

In [ ]:
def gradient_descent(x, y, learning_rate, num_iterations, return_history=False):
    b, m = 0.0, 0.0
    history = []
    x = list(x) if not isinstance(x, list) else x
    y = list(y) if not isinstance(y, list) else y
    for _ in range(num_iterations):
        b, m = step_gradient(x, y, b, m, learning_rate)
        if return_history:
            yhat = [m*xi + b for xi in x]
            loss = sum((yi-yh)**2 for yi,yh in zip(y,yhat)) / len(y)
            history.append((b, m, loss))
    if return_history:
        return [b, m], history
    return [b, m]

# Quick demo on the short quarterly series
b_q, m_q = gradient_descent(ad_spend, sales_obs, 0.0005, 3000)
print(f'Quarterly GD → m={m_q:.3f}, b={b_q:.2f}')

## 8. From-Scratch GD on Advertising → Sales (centered)

In [ ]:
X = df['advertising'].values.astype(float)
y = df['sales'].values.astype(float)

Xc = X - X.mean()
yc = y - y.mean()

(b_c, m_c), hist = gradient_descent(Xc.tolist(), yc.tolist(),
                                    learning_rate=0.0005,
                                    num_iterations=2500,
                                    return_history=True)
m_gd = m_c
b_gd = y.mean() - m_gd * X.mean()
print(f'Centered GD → m={m_gd:.4f}, b={b_gd:.2f}')
print(f'Marginal return to advertising: ${m_gd:.2f}k sales per $1k ad spend')

yhat_gd = m_gd * X + b_gd
mse_gd = np.mean((y - yhat_gd)**2)
ss_tot = np.sum((y - y.mean())**2)
r2_gd = 1 - np.sum((y - yhat_gd)**2) / ss_tot
print(f'MSE={mse_gd:.1f}, R²={r2_gd:.4f}')

plt.figure(figsize=(7,5))
plt.scatter(X, y, alpha=0.55, edgecolor='k', linewidth=0.3, c='#2980B9', label='Data')
xx = np.linspace(X.min()-2, X.max()+2, 100)
plt.plot(xx, m_gd*xx + b_gd, 'r-', lw=2.2, label=f'GD: sales={m_gd:.2f}·ad+{b_gd:.1f}')
plt.xlabel('Advertising ($000)'); plt.ylabel('Sales ($000)')
plt.title('From-Scratch Gradient Descent Fit')
plt.legend(); plt.tight_layout(); plt.show()

losses = [h[2] for h in hist]
plt.figure(figsize=(6,3.5))
plt.plot(losses); plt.yscale('log')
plt.xlabel('Iteration'); plt.ylabel('MSE (centered)')
plt.title('Convergence of Centered GD'); plt.tight_layout(); plt.show()

## 9. scikit-learn LinearRegression

In [ ]:
X_2d = X.reshape(-1, 1)
model = LinearRegression().fit(X_2d, y)
m_sk, b_sk = model.coef_[0], model.intercept_
r2_sk = model.score(X_2d, y)
print(f'sklearn → m={m_sk:.4f}, b={b_sk:.2f}, R²={r2_sk:.4f}')

yhat_sk = model.predict(X_2d)

plt.figure(figsize=(7,5))
plt.scatter(X, y, alpha=0.55, edgecolor='k', linewidth=0.3, c='#2980B9', label='Data')
plt.plot(xx, m_sk*xx + b_sk, 'g-', lw=2.2, label=f'sklearn: sales={m_sk:.2f}·ad+{b_sk:.1f}')
plt.xlabel('Advertising ($000)'); plt.ylabel('Sales ($000)')
plt.title('scikit-learn LinearRegression Fit')
plt.legend(); plt.tight_layout(); plt.show()

## 10. More Practice — Closed Form, Residuals, Consumption Function

In [ ]:
# Closed-form OLS
x_bar, y_bar = X.mean(), y.mean()
m_cf = np.sum((X - x_bar)*(y - y_bar)) / np.sum((X - x_bar)**2)
b_cf = y_bar - m_cf * x_bar
print(f'Closed-form → m={m_cf:.4f}, b={b_cf:.2f}')

comparison = pd.DataFrame({
    'Method': ['Centered GD', 'sklearn', 'Closed-form'],
    'Slope m (marginal return)': [m_gd, m_sk, m_cf],
    'Intercept b': [b_gd, b_sk, b_cf]
})
display(comparison.round(4))

# Residual diagnostics
resid = y - yhat_sk
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(yhat_sk, resid, alpha=0.55, edgecolor='k', linewidth=0.3, c='#2980B9')
axes[0].axhline(0, color='r', ls='--')
axes[0].set_xlabel('Fitted sales ($000)'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Fitted')
axes[1].hist(resid, bins=20, edgecolor='k', alpha=0.75, color='#2980B9')
axes[1].set_xlabel('Residual'); axes[1].set_title('Residual Distribution')
plt.suptitle(f'Residual Analysis (R² = {r2_sk:.3f})')
plt.tight_layout(); plt.show()
print('Residual mean ≈ 0?', round(resid.mean(), 6))

# --- Consumption function (Keynesian) ---
cons = pd.read_csv('data/economics_consumption.csv')
Yd = cons['disposable_income'].values
C  = cons['consumption'].values
mpc_model = LinearRegression().fit(Yd.reshape(-1,1), C)
mpc = mpc_model.coef_[0]
autonomous = mpc_model.intercept_
r2_c = mpc_model.score(Yd.reshape(-1,1), C)
print(f'\nConsumption function: C = {autonomous:.2f} + {mpc:.3f}·Yd')
print(f'Estimated MPC = {mpc:.3f}  |  R² = {r2_c:.3f}')
print('Policy reading: households spend about {:.0f} cents of each extra dollar of disposable income.'.format(mpc*100))

plt.figure(figsize=(7,4.5))
plt.scatter(Yd, C, alpha=0.6, edgecolor='k', linewidth=0.3, c='#27AE60')
xxc = np.linspace(Yd.min(), Yd.max(), 80)
plt.plot(xxc, autonomous + mpc*xxc, 'r-', lw=2, label=f'C = {autonomous:.1f} + {mpc:.2f}·Yd')
plt.xlabel('Disposable income ($000)'); plt.ylabel('Consumption ($000)')
plt.title('Keynesian Consumption Function (estimated MPC)')
plt.legend(); plt.tight_layout(); plt.show()

## 11. Simulation — Learning Rate, Noise & Sample Size

In [ ]:
# === TUNABLE PARAMETERS ===
LEARNING_RATE = 0.0005
N_ITER        = 2500
NOISE_STD     = 0.0
SAMPLE_FRAC   = 1.0
N_REPS        = 30
# ===========================

def run_one(X, y, alpha, n_iter, noise_std, sample_frac):
    n = len(X)
    idx = np.random.choice(n, size=max(20, int(n*sample_frac)), replace=False)
    Xs, ys = X[idx], y[idx].copy()
    if noise_std > 0:
        ys = ys + np.random.normal(0, noise_std, size=len(ys))
    Xc, yc = Xs - Xs.mean(), ys - ys.mean()
    b_c, m_c = gradient_descent(Xc.tolist(), yc.tolist(), alpha, n_iter)
    m = m_c
    b = ys.mean() - m * Xs.mean()
    yhat = m*Xs + b
    r2 = 1 - np.sum((ys-yhat)**2) / np.sum((ys-ys.mean())**2)
    return m, b, r2

ms, bs, r2s = [], [], []
for _ in range(N_REPS):
    m, b, r2 = run_one(X, y, LEARNING_RATE, N_ITER, NOISE_STD, SAMPLE_FRAC)
    ms.append(m); bs.append(b); r2s.append(r2)

print(f'Settings: α={LEARNING_RATE}, iters={N_ITER}, noise={NOISE_STD}, frac={SAMPLE_FRAC}')
print(f'Slope (marginal return)  mean±std : {np.mean(ms):.4f} ± {np.std(ms):.4f}')
print(f'Intercept                mean±std : {np.mean(bs):.2f} ± {np.std(bs):.2f}')
print(f'R²                       mean±std : {np.mean(r2s):.4f} ± {np.std(r2s):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].hist(ms, bins=12, edgecolor='k', alpha=0.75, color='#2980B9')
axes[0].axvline(m_sk, color='r', ls='--', label='sklearn'); axes[0].legend(fontsize=8)
axes[0].set_title('Slope (marginal return)')
axes[1].hist(bs, bins=12, edgecolor='k', alpha=0.75, color='#2980B9')
axes[1].axvline(b_sk, color='r', ls='--'); axes[1].set_title('Intercept')
axes[2].hist(r2s, bins=12, edgecolor='k', alpha=0.75, color='#2980B9')
axes[2].axvline(r2_sk, color='r', ls='--'); axes[2].set_title('R²')
plt.suptitle('Monte-Carlo Sensitivity — Advertising → Sales')
plt.tight_layout(); plt.show()

### Extra: learning-rate effect on loss trajectory

In [ ]:
def loss_trajectory(X, y, alpha, n_iter=2000):
    Xc, yc = X - X.mean(), y - y.mean()
    b, m = 0.0, 0.0
    losses = []
    for _ in range(n_iter):
        b, m = step_gradient(Xc.tolist(), yc.tolist(), b, m, alpha)
        err = yc - (m*Xc + b)
        losses.append(np.mean(err**2))
    return losses

plt.figure(figsize=(8,4))
for alpha in [1e-5, 5e-5, 1e-4, 5e-4, 0.001]:
    losses = loss_trajectory(X, y, alpha)
    plt.plot(losses, label=f'α={alpha}')
plt.yscale('log'); plt.xlabel('Iteration'); plt.ylabel('MSE (centered)')
plt.title('Learning-Rate Effect — Economics GD')
plt.legend(); plt.tight_layout(); plt.show()

## Cheat Sheet

| Concept | Formula / Code | Economics reading |
|---------|----------------|-------------------|
| Line | `yhat = m*x + b` | Predicted sales or consumption |
| Slope `m` | marginal effect | Extra sales per extra $1k ad; or MPC |
| Intercept `b` | baseline | Use cautiously outside observed range |
| SSE / MSE | `sum((y-yhat)**2)` / mean | Total / average squared prediction error |
| Gradient b / m | `−2/N · Σ …` | Direction of steepest ascent of loss |
| Update | `param -= α * gradient` | Step downhill on the loss surface |
| Convergence | params almost stop changing | Approximate minimum of loss reached |
| sklearn | `LinearRegression().fit` | Fast reference OLS |
| Closed form | `m = cov(x,y)/var(x)` | Exact simple-OLS solution |
| Centering | GD on demeaned data | Numerically stable when scales differ |

**Key results (advertising → sales)**
- Marginal return ≈ **$2.43k sales per $1k advertising**
- Intercept ≈ **$55.6k**
- R² ≈ **0.55**
- GD, sklearn and closed-form agree closely when learning rate and iterations are sensible.

**Consumption function (practice)**
- Estimated MPC ≈ **0.73** (households spend ~73 ¢ of each extra dollar of disposable income).
